<a href="https://colab.research.google.com/github/arpita-png/pattern-recognition-project/blob/main/pattern_rec_assg_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
"""
Implement K-Means Clustering on the Boston Housing dataset

See here for more details on the dataset - https://www.kaggle.com/code/prasadperera/the-boston-housing-dataset.
"""

import random


def load_data():
    """
    Load the data

    Note: Assumes that data is in current working directory
    """
    X, Y = [], []

    with open('BostonHousing-1.csv', "r") as file:
        for ix, line in enumerate(file):
            row = line.split(",")
            row = [r.replace('"', "").strip() for r in row]

            if ix == 0:
                columns = row

            elif row:
                X.append(row[:-1])
                Y.append(row[-1])

    return X, Y, columns


# TODO
def fit(X, K):
    """
    Find the K prototypes

    return: K prototypes
    """

    X_float = []
    for row in X:
        X_float.append([float(value) for value in row])

    random.seed(42)
    max_iterations = 100

    prototypes = random.sample(X_float, K)

    for iteration in range(max_iterations):
        sample_clusters = assign_samples(X_float, prototypes)

        new_prototypes = []

        for k in range(K):
            cluster_points = []

            for i in range(len(X_float)):
                if sample_clusters[i] == k:
                    cluster_points.append(X_float[i])

            if len(cluster_points) == 0:
                new_prototypes.append(random.choice(X_float))
            else:
                prototype = []

                for feature_index in range(len(X_float[0])):
                    total = 0

                    for point in cluster_points:
                        total += point[feature_index]

                    mean_value = total / len(cluster_points)
                    prototype.append(mean_value)

                new_prototypes.append(prototype)

        if new_prototypes == prototypes:
            break

        prototypes = new_prototypes

    return prototypes


# TODO
def assign_samples(X, prototypes):
    """
    Assign each sample to one of the K prototypes

    return: assigned cluster for each sample
    """

    sample_clusters = []

    for row in X:
        sample = [float(value) for value in row]

        distances = []

        for prototype in prototypes:
            distance = 0

            for i in range(len(sample)):
                distance += (sample[i] - prototype[i]) ** 2

            distances.append(distance)

        closest_cluster = distances.index(min(distances))
        sample_clusters.append(closest_cluster)

    return sample_clusters


# TODO
def mean_squared_error(X, prototypes, sample_clusters):
    """
    Get the inner-cluster mean squared error of the model on the data
    """

    total_error = 0

    for i in range(len(X)):
        sample = [float(value) for value in X[i]]
        cluster = sample_clusters[i]
        prototype = prototypes[cluster]

        distance = 0

        for j in range(len(sample)):
            distance += (sample[j] - prototype[j]) ** 2

        total_error += distance

    mse = total_error / len(X)

    return mse


def main():
    # Load data
    X, _, _ = load_data()

    for k in range(1, 9):
        # TODO
        prototypes = fit(X, k)
        sample_cluster = assign_samples(X, prototypes)

        # TODO
        print(f"\nModel MSE for {k}:", mean_squared_error(X, prototypes, sample_cluster))



if __name__ == "__main__":
    main()


Model MSE for 1: 38257.60409324616

Model MSE for 2: 11323.401602052121

Model MSE for 3: 8749.402608496943

Model MSE for 4: 8272.204203714333

Model MSE for 5: 7753.740764245256

Model MSE for 6: 7565.3834925328465

Model MSE for 7: 6855.999995037598

Model MSE for 8: 6707.271941316578


In [6]:
"""
Implement PCA on the Boston Housing dataset. Then fit linear regression on
the transformed data.

See here for more details on the dataset - https://www.kaggle.com/code/prasadperera/the-boston-housing-dataset.
"""

import numpy as np


X_mean = None
X_std = None


def load_data():
    """
    Load the data

    Note: Assumes that data is in current working directory
    """
    X, Y = [], []

    with open('BostonHousing-1.csv', "r") as file:
        for ix, line in enumerate(file):
            row = line.split(",")
            row = [r.replace('"', "").strip() for r in row]

            if ix == 0:
                columns = row

            elif row:
                X.append(row[:-1])
                Y.append(row[-1])

    return X, Y, columns


# TODO
def find_principal_comps_and_singular_values(X, K):
    """
    Find the top K principal components and singular values

    NOTE: You need to calculate the total sum of singular values ("total_singular") for
    calculation of the % of explained variance
    """

    global X_mean, X_std

    X = np.array(X, dtype=float)

    X_mean = np.mean(X, axis=0)
    X_std = np.std(X, axis=0)

    X_std[X_std == 0] = 1

    X_scaled = (X - X_mean) /X_std

    U, S, Vt = np.linalg.svd(X_scaled, full_matrices=False)

    principal_comps = Vt

    top_k_comps = principal_comps[:K]

    singular_variance = S ** 2

    top_k_singular =np.sum(singular_variance[:K])
    total_singular =np.sum(singular_variance)

    return top_k_comps, top_k_singular, total_singular


# TODO
def transform_data(X, principal_comps):
    """
    Transform the data using the top K principal components
    """

    global X_mean, X_std

    X = np.array(X, dtype=float)

    X_scaled = (X - X_mean)/X_std

    X_k = np.dot(X_scaled, principal_comps.T)

    return X_k


# TODO
def fit(X_k, Y):
    """
    Fit linear regression
    """

    X_k = np.array(X_k, dtype= float)
    Y = np.array(Y, dtype= float)

    ones = np.ones((X_k.shape[0], 1))
    X_bias = np.hstack((ones, X_k))

    weights = np.linalg.pinv(X_bias).dot(Y)

    return weights


# TODO
def mean_squared_error(X, Y, weights):
    """
    Get the mean squared error of the model on the data
    """

    X = np.array(X, dtype=float)
    Y = np.array(Y, dtype=float)

    ones = np.ones((X.shape[0], 1))
    X_bias = np.hstack((ones, X))

    predictions = X_bias.dot(weights)

    mse = np.mean((Y - predictions) ** 2)

    return mse


def main():
    # Load data
    X, Y, columns = load_data()

    for k in range(1, 9):
        # TODO
        top_k_comps, top_k_singular, total_singular = find_principal_comps_and_singular_values(X, k)
        X_k = transform_data(X, top_k_comps)
        print(f"\n% of Explained Variance for {k}:", top_k_singular / total_singular)

        # TODO
        model = fit(X_k, Y)
        print(f"\nModel MSE for {k}:", mean_squared_error(X_k, Y, model))



if __name__ == "__main__":
    main()


% of Explained Variance for 1: 0.47129606357274684

Model MSE for 1: 52.827050382005666

% of Explained Variance for 2: 0.581547996048627

Model MSE for 2: 45.935685239956236

% of Explained Variance for 3: 0.6771338939748566

Model MSE for 3: 30.73514536584494

% of Explained Variance for 4: 0.7431012099832176

Model MSE for 4: 29.735581982325783

% of Explained Variance for 5: 0.8073178205045912

Model MSE for 5: 25.5808626899439

% of Explained Variance for 6: 0.857887603227504

Model MSE for 6: 25.285779794329834

% of Explained Variance for 7: 0.8990688406240485

Model MSE for 7: 25.28102708734357

% of Explained Variance for 8: 0.9295378648139051

Model MSE for 8: 24.85330548637312
